# Valutazione finale: tutti gli esperimenti rispetto al top-down

La tabella riepilogativa contiene Precision, Recall e F1 medie di HLG e LLG per il top-down originale, il bottom-up originale e ogni configurazione della campagna selezionata. Per default viene scelta la suite piu recente; SUITE_ID permette di selezionarne un'altra senza mescolare versioni.

Si valuta la gerarchia finale dopo tutte le iterazioni di ciascuna run completata, anche senza convergenza. Non si mediano gli stati intermedi. Per ogni dataset si mediano prima le ripetizioni finali; poi si calcola separatamente la media aritmetica di Precision, Recall e F1 tra i dataset, con lo stesso peso per ciascuno. La F1 media e la media delle F1, non la F1 ricavata da Precision e Recall medie.

I metodi con risultati disponibili vengono confrontati sugli stessi dataset e sulle stesse baseline. I metodi ancora senza risultati rimangono in tabella con metriche mancanti e non restringono la coorte. Le colonne di copertura segnalano campagne parziali: la tabella definitiva richiede il completamento delle run previste. Nessuna generazione viene avviata e nessun risultato mancante viene sostituito con zero.

Una seconda tabella unica riporta ogni esperimento per dataset, con Precision, Recall e F1 di HLG e LLG e i sei delta rispetto alla baseline top-down corrispondente. Usa tutti i risultati finali disponibili, anche fuori dalla coorte comune del riepilogo; le ripetizioni sono mediate entro dataset.

Una terza tabella mostra convergenza, iterazioni, motivo di arresto e conteggi HLG/LLG top-down e bottom-up per ciascuna run, mantenendo separate le ripetizioni.

Una quarta tabella conta le decisioni valide di ciascun LLM per esperimento, dataset e ripetizione, con esempi, temperatura e diagnostica dei tentativi scartati.

La configurazione della suite con **4 esempi e temperatura 0,3** viene esclusa da metriche, decisioni ed export: il confronto mantiene gli output del notebook 02 come bottom-up di riferimento. Le altre quattro configurazioni della suite restano disponibili. I file sperimentali salvati non vengono modificati.


In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir() and (PROJECT_ROOT.parent / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
for directory in (PROJECT_ROOT, PROJECT_ROOT / "notebook"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

from groundtruth import GROUNDTRUTH_BY_NAME

# None seleziona la suite piu recente; impostare un ID per una campagna specifica.
SUITE_ID = None
EXPERIMENTS_ROOT = PROJECT_ROOT / "output" / "bottom_up_experiments"
available_suites = sorted(p.parent for p in EXPERIMENTS_ROOT.glob("*/manifest.json"))
if not available_suites:
    raise ValueError("Nessuna suite sperimentale salvata")
SUITE_DIRS = [EXPERIMENTS_ROOT / SUITE_ID] if SUITE_ID else [available_suites[-1]]
REPORT_ID = datetime.now(ZoneInfo("Europe/Rome")).strftime("%Y%m%d_%H%M%S_%f")
REPORT_DIR = PROJECT_ROOT / "output" / "evaluation" / "configuration_comparison" / REPORT_ID
print("Suite:", SUITE_DIRS[0].name)
print("Report:", REPORT_DIR)


## Lettura degli output salvati


In [ ]:
from collections import Counter
import hashlib
import json
from pathlib import Path


def digest(raw):
    return hashlib.sha256(raw).hexdigest()


def baseline_key(payload):
    fields = ("description", "structuredHighLevelGoals", "structuredLowLevelGoals")
    if any(field not in payload for field in fields):
        raise ValueError("Missing verifiable initial hierarchy or description")
    return digest(json.dumps({field: payload[field] for field in fields},
                             sort_keys=True, ensure_ascii=False).encode("utf-8"))


def goal_texts(container, field):
    from src.evaluation.goal_evaluator import goal_descriptions
    return goal_descriptions(container, field)




In [ ]:
def collect_outputs(project_root, suite_directories):
    """Keep planned/missing runs visible; read actual JSON results, never old CSVs."""
    # Notebook kernels retain imported modules after source files are updated.
    # Refresh the shared schema before storage so both use the current contract.
    import importlib
    import src.data_model as data_model
    import src.bottom_up.result_storage as result_storage
    importlib.invalidate_caches()
    importlib.reload(data_model)
    importlib.reload(result_storage)
    from src.bottom_up.result_storage import payload_hash, result_payload, require_current_schema
    root = Path(project_root)
    hashes, entries, availability, baselines = {}, [], [], {}
    configurations = []

    def read(path):
        path = Path(path).resolve()
        raw = path.read_bytes()
        checksum = digest(raw)
        if path in hashes and hashes[path] != checksum:
            raise ValueError(f"Input changed during collection: {path}")
        hashes[path] = checksum
        return json.loads(raw)

    def register_baseline(payload, path):
        key = baseline_key(payload)
        identity = (payload["name"], key)
        if identity not in baselines:
            entry = {"dataset": payload["name"], "baseline_key": key, "suite_id": "",
                     "method": "top_down", "config_id": "top_down", "repetition": 0,
                     "source_file": str(path), "payload": payload,
                     "hlg": payload["structuredHighLevelGoals"], "llg": payload["structuredLowLevelGoals"]}
            goal_texts(entry["hlg"], "goals")
            goal_texts(entry["llg"], "low_level_goals")
            entries.append(entry)
            baselines[identity] = entry
            availability.append({k: v for k, v in entry.items() if k not in {"payload", "hlg", "llg"}} | {"status": "completed"})
        return key

    for path in sorted((root / "output" / "top_down").glob("*_top_down.json")):
        register_baseline(read(path), path)
    for directory in map(Path, suite_directories):
        manifest = read(directory / "manifest.json")
        require_current_schema(manifest)
        if manifest.get("storage_layout") != "dataset_json_v3":
            raise ValueError("Unsupported suite layout; prepare a new schema-3 suite")
        # The 4-example / 0.3 condition is represented by notebook 02 outputs.
        # Exclude its suite copy from all metrics, decisions and exports.
        selected_configs = [config for config in manifest["configurations"]
                            if not (config.get("evaluator_few_shot_examples") == 4
                                    and config.get("evaluator_temperature") == 0.3)]
        configurations.extend({"suite_id": manifest["suite_id"],
                               "design": manifest.get("design", "unspecified"),
                               "planned_datasets": len(manifest["baselines"]),
                               "planned_repetitions": manifest["repetitions"], **config}
                              for config in selected_configs)
        for baseline in manifest["baselines"]:
            payload = baseline["payload"]
            if payload_hash(payload) != baseline["sha256"]:
                raise ValueError("Frozen compact baseline changed")
            key = register_baseline(payload, directory / "manifest.json")
            path = directory / baseline["dataset_file"]
            document = require_current_schema(read(path)) if path.exists() else None
            if document and (document["baseline_sha256"] != baseline["sha256"]
                             or payload_hash(document["baseline"]) != baseline["sha256"]):
                raise ValueError("Compact dataset baseline differs from the manifest")
            for config in selected_configs:
                for repetition in range(1, manifest["repetitions"] + 1):
                    run_id = f"{config['config_id']}/repeat_{repetition:03d}"
                    record = document["runs"].get(run_id, {}) if document else {}
                    identity = dict(dataset=baseline["dataset"], baseline_key=key,
                        suite_id=manifest["suite_id"], config_id=config["config_id"],
                        method=f"{manifest['suite_id']}/{config['config_id']}", repetition=repetition)
                    row = {**identity, "status": record.get("status", "not_started"),
                           "source_file": str(path), "error": record.get("error")}
                    if row["status"] == "completed":
                        if record["experiment"] != config or record["repetition"] != repetition:
                            raise ValueError("Compact run configuration differs from the manifest")
                        result = result_payload(document, record)
                        goal_texts(result["finalHighLevelGoals"], "goals")
                        goal_texts(result["finalLowLevelGoals"], "low_level_goals")
                        entries.append({**identity, "source_file": str(path), "payload": result,
                                        "hlg": result["finalHighLevelGoals"], "llg": result["finalLowLevelGoals"]})
                    availability.append(row)
    ordinary_root = root / "output" / "top_down_bottom_up"
    batches = sorted(path for path in (ordinary_root / "runs").glob("*")
                     if path.is_dir() and (path / "results_summary.json").is_file())
    # Notebook 02 saves isolated batches; evaluate the latest started batch only.
    # Legacy files are never combined with a new batch or converted automatically.
    ordinary_directory = batches[-1] if batches else ordinary_root
    for path in sorted((ordinary_directory / "datasets").glob("*.json")):
        document = require_current_schema(read(path))
        if payload_hash(document["baseline"]) != document["baseline_sha256"]:
            raise ValueError("Compact ordinary baseline altered")
        for record in document["runs"].values():
            identity = dict(dataset=document["name"], method="bottom_up", config_id="bottom_up",
                            suite_id="", repetition=record.get("repetition", 1), source_file=str(path))
            if record.get("status") != "completed":
                availability.append({**identity, "status": record.get("status", "not_started"),
                                     "error": record.get("error")})
                continue
            key = register_baseline(document["baseline"], path)
            result = result_payload(document, record)
            entries.append({**identity, "baseline_key": key, "payload": result,
                            "hlg": result["finalHighLevelGoals"], "llg": result["finalLowLevelGoals"]})
            availability.append({**identity, "baseline_key": key, "status": "completed"})
    for (dataset, key) in baselines:
        if not any(e["method"] == "bottom_up" and e["dataset"] == dataset
                   and e["baseline_key"] == key for e in entries):
            availability.append({"dataset": dataset, "baseline_key": key,
                "method": "bottom_up", "config_id": "bottom_up",
                "suite_id": "", "repetition": 0, "status": "unavailable",
                "error": "No eligible original bottom-up output for this baseline"})
    return {"entries": entries, "availability": availability, "input_hashes": hashes,
            "configurations": configurations}




In [ ]:
METRICS = ["precision", "recall", "f1"]


def evaluate_final_outputs(collection, references, score_goals):
    """Score each final hierarchy once; intermediate iterations are never averaged."""
    rows, matches = [], []
    for entry in collection["entries"]:
        identity = {k: entry[k] for k in (
            "dataset", "baseline_key", "suite_id", "method", "config_id", "repetition", "source_file")}
        if entry["dataset"] not in references:
            raise ValueError(f"Missing ground truth: {entry['dataset']}")
        cycle = entry["payload"].get("globalGoalCycle", {})
        for level, field, ref_field in (("HLG", "goals", "highLevelGoals"),
                                       ("LLG", "low_level_goals", "lowLevelGoals")):
            predictions = goal_texts(entry[level.lower()], field)
            reference = references[entry["dataset"]][ref_field]
            result = dict(score_goals(predictions, reference))
            assigned = result.pop("matches", [])
            rows.append({**identity, "level": level,
                "precision": result["precision"], "recall": result["recall"],
                "f1": result.get("f1", result.get("f1_score")),
                "generated_goal_count": len(predictions), "reference_goal_count": len(reference),
                "completed_iterations": cycle.get("completed_iterations"),
                "converged": cycle.get("converged"), "stop_reason": cycle.get("stop_reason")})
            matches.extend({**identity, "level": level, **match} for match in assigned)
    columns = ["dataset", "baseline_key", "suite_id", "method", "config_id", "repetition",
               "source_file", "level", *METRICS, "generated_goal_count", "reference_goal_count",
               "completed_iterations", "converged", "stop_reason"]
    return pd.DataFrame(rows, columns=columns), pd.DataFrame(matches)


def final_comparison_table(metrics, configurations, availability):
    """Macro means over final results on a common cohort, repetitions averaged first."""
    methods = ["top_down", "bottom_up"] + list(dict.fromkeys(
        f"{c['suite_id']}/{c['config_id']}" for c in configurations))
    datasets = metrics.groupby(["method", "dataset", "baseline_key", "level"], dropna=False)[METRICS].mean().reset_index()
    # A dataset is eligible only when both HLG and LLG were evaluated.
    cohorts = {}
    for method in methods:
        group = datasets[datasets.method == method]
        counts = group.groupby(["dataset", "baseline_key"])["level"].nunique()
        cohorts[method] = set(counts[counts == 2].index)
    active = [method for method in methods if method != "top_down" and cohorts[method]]
    common = set(cohorts["top_down"])
    for method in active:
        common &= cohorts[method]
    if common:
        names = [name for name, _ in common]
        if len(names) != len(set(names)):
            raise ValueError("Multiple baseline versions for one dataset; select one compatible suite")
    mask = pd.Series([tuple(row) in common for row in datasets[["dataset", "baseline_key"]].to_numpy()], index=datasets.index, dtype=bool)
    selected = datasets.loc[mask]
    configuration_map = {f"{c['suite_id']}/{c['config_id']}": c for c in configurations}
    rows = []
    for method in methods:
        frame = selected[selected.method == method]
        raw = metrics[(metrics.method == method) & metrics.level.eq("HLG")]
        runs_used = sum((d, b) in common for d, b in raw[["dataset", "baseline_key"]].itertuples(index=False, name=None))
        row = {"Esperimento": "Top-down originale" if method == "top_down" else (
            "Bottom-up originale" if method == "bottom_up" else method.split("/", 1)[1]),
            "Dataset valutati": len(common & cohorts[method]),
            "Dataset disponibili": len(cohorts[method]), "Run valutate": runs_used}
        for level in ("HLG", "LLG"):
            averages = frame[frame.level == level][METRICS].mean()
            for metric, label in (("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")):
                row[f"{level} {label}"] = averages[metric]
        config = configuration_map.get(method, {})
        row["Esempi evaluator"] = config.get("evaluator_few_shot_examples")
        row["Temperatura evaluator"] = config.get("evaluator_temperature")
        planned = [r for r in availability if r.get("method") == method]
        row["Stato"] = ("Nessun risultato finale" if not cohorts[method] else
            "Nessun dataset comune" if not common else
            "Parziale" if any(r.get("status") != "completed" for r in planned) else "Disponibile")
        rows.append(row)
    columns = ["Esperimento", "Esempi evaluator", "Temperatura evaluator", "HLG Precision", "HLG Recall", "HLG F1", "LLG Precision", "LLG Recall", "LLG F1",
               "Dataset valutati", "Dataset disponibili", "Run valutate", "Stato"]
    table = pd.DataFrame(rows, columns=columns)
    for level in ("HLG", "LLG"):
        for metric in ("Precision", "Recall", "F1"):
            column = f"{level} {metric}"
            table[f"{level} Delta {metric}"] = table[column] - table.loc[0, column]
    metric_columns = [column for level in ("HLG", "LLG")
                      for column in ([f"{level} {m}" for m in ("Precision", "Recall", "F1")]
                                     + [f"{level} Delta {m}" for m in ("Precision", "Recall", "F1")])]
    return table[["Esperimento", "Esempi evaluator", "Temperatura evaluator", *metric_columns,
                  "Dataset valutati", "Dataset disponibili", "Run valutate", "Stato"]], common


def dataset_comparison_table(metrics, configurations, availability):
    """One row per method/dataset/baseline, paired with its own top-down result."""
    methods = ["top_down", "bottom_up"] + list(dict.fromkeys(
        f"{c['suite_id']}/{c['config_id']}" for c in configurations))
    configs = {f"{c['suite_id']}/{c['config_id']}": c for c in configurations}
    identity = ["method", "dataset", "baseline_key", "level"]
    means = metrics.groupby(identity, dropna=False)[METRICS].mean()
    counts = metrics.groupby(identity, dropna=False)["repetition"].nunique()
    keys = set(metrics[["dataset", "baseline_key"]].itertuples(index=False, name=None))
    keys.update((r["dataset"], r["baseline_key"]) for r in availability
                if r.get("baseline_key") is not None)
    rows = []
    for method in methods:
        config = configs.get(method, {})
        for dataset, baseline in sorted(keys):
            planned = [r for r in availability if r.get("method") == method
                       and r.get("dataset") == dataset and r.get("baseline_key") == baseline]
            row = {"Dataset": dataset, "Esperimento": "Top-down originale" if method == "top_down" else (
                "Bottom-up originale" if method == "bottom_up" else method.split("/", 1)[1]),
                "Suite": config.get("suite_id", ""), "Baseline": baseline,
                "Esempi evaluator": config.get("evaluator_few_shot_examples"),
                "Temperatura evaluator": config.get("evaluator_temperature")}
            level_counts, paired = [], True
            for level in ("HLG", "LLG"):
                key = (method, dataset, baseline, level)
                reference = ("top_down", dataset, baseline, level)
                level_counts.append(int(counts.get(key, 0)))
                paired = paired and reference in means.index
                for metric, label in (("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")):
                    value = means.loc[key, metric] if key in means.index else float("nan")
                    base = means.loc[reference, metric] if reference in means.index else float("nan")
                    row[f"{level} {label}"] = value
                    row[f"{level} Delta {label}"] = value - base
            row["Ripetizioni valutate"] = min(level_counts)
            expected = config.get("planned_repetitions", 1)
            row["Stato"] = ("Nessun risultato finale" if not any(level_counts) else
                "Baseline top-down non disponibile" if not paired else
                "Parziale" if min(level_counts) < expected or len(set(level_counts)) > 1
                or any(r.get("status") != "completed" for r in planned) else "Disponibile")
            rows.append(row)
    metric_columns = [column for level in ("HLG", "LLG")
                      for column in ([f"{level} {m}" for m in ("Precision", "Recall", "F1")]
                                     + [f"{level} Delta {m}" for m in ("Precision", "Recall", "F1")])]
    return pd.DataFrame(rows, columns=["Esperimento", "Dataset", "Esempi evaluator",
        "Temperatura evaluator", *metric_columns, "Ripetizioni valutate", "Stato", "Suite", "Baseline"])


def convergence_goal_counts_table(collection):
    """Report final counts and convergence per run, without averaging repetitions."""
    def run_key(record):
        return tuple(record.get(field) for field in
                     ("method", "dataset", "baseline_key", "repetition"))

    baselines = {(e["dataset"], e["baseline_key"]): e
                 for e in collection["entries"] if e["method"] == "top_down"}
    entries = {run_key(e): e for e in collection["entries"] if e["method"] != "top_down"}
    runs = {run_key(r): r for r in collection["availability"] if r["method"] != "top_down"}
    for key, entry in entries.items():
        runs.setdefault(key, {**entry, "status": "completed"})
    rows = []
    for key, record in sorted(runs.items(), key=lambda item: tuple(str(v) for v in item[0])):
        method, dataset, baseline_key, repetition = key
        entry = entries.get(key)
        baseline = baselines.get((dataset, baseline_key))
        cycle = entry["payload"].get("globalGoalCycle", {}) if entry else {}
        converged = cycle.get("converged")
        row = {"Esperimento": "Bottom-up originale" if method == "bottom_up" else method.split("/", 1)[-1],
               "Dataset": dataset, "Ripetizione": repetition,
               "Stato run": record.get("status", "unavailable"),
               "Convergenza": "Si" if converged is True else "No" if converged is False else "Non disponibile",
               "Iterazioni completate": cycle.get("completed_iterations"),
               "Motivo arresto": cycle.get("stop_reason"),
               "Suite": record.get("suite_id", ""), "Baseline": baseline_key}
        for level, field in (("HLG", "goals"), ("LLG", "low_level_goals")):
            initial = len(goal_texts(baseline[level.lower()], field)) if baseline else None
            final = len(goal_texts(entry[level.lower()], field)) if entry else None
            row[f"{level} top-down"] = initial
            row[f"{level} bottom-up"] = final
            row[f"Delta numero {level}"] = final - initial if initial is not None and final is not None else None
        rows.append(row)
    return pd.DataFrame(rows, columns=["Esperimento", "Dataset", "Ripetizione", "Stato run",
        "Convergenza", "Iterazioni completate", "Motivo arresto", "HLG top-down", "HLG bottom-up",
        "Delta numero HLG", "LLG top-down", "LLG bottom-up", "Delta numero LLG", "Suite", "Baseline"])


def evaluator_decision_counts_table(collection):
    """Count accepted global-response actions once per model/iteration/branch."""
    actions = ["KEEP_ORIGINAL_HIGH_LEVEL_GOAL", "REWRITE_ORIGINAL_HIGH_LEVEL_GOAL",
               "REMOVE_ORIGINAL_HIGH_LEVEL_GOAL", "DUPLICATE_REMOVAL"]
    labels = dict(zip(actions, ["KEEP", "REWRITE", "REMOVE", "DUPLICATE_REMOVAL"]))
    configs = {f"{c['suite_id']}/{c['config_id']}": c for c in collection["configurations"]}
    rows = []
    for entry in collection["entries"]:
        if entry["method"] == "top_down":
            continue
        cycle = entry["payload"].get("globalGoalCycle", {})
        settings = {**configs.get(entry["method"], {}), **cycle.get("configuration", {})}
        traces = [t for t in cycle.get("iterations", []) if t.get("checkpoint_complete", True)]
        models = list(dict.fromkeys(list(settings.get("evaluator_models", [])) +
            [v["model"] for t in traces for v in t.get("evaluator_responses", [])]))
        for model in models:
            counts = {action: 0 for action in actions}
            valid, invalid, rejected_attempts, missing, other = 0, 0, 0, 0, 0
            for trace in traces:
                records = [v for v in trace.get("evaluator_responses", []) if v["model"] == model]
                if len(records) > 1:
                    raise ValueError(f"Duplicate global vote for {model}, iteration {trace['iteration']}")
                if not records:
                    missing += 1
                    continue
                record = records[0]
                rejected_attempts += sum(a.get("generation", {}).get("accepted") is False
                                         for a in record.get("attempts", []))
                if record.get("status") != "VALID":
                    invalid += 1
                    continue
                valid += 1
                decisions = record.get("decisions", [])
                ids = [item["branch_id"] for item in decisions]
                if len(ids) != len(set(ids)):
                    raise ValueError("Repeated branch IDs in a saved accepted global response")
                for item in decisions:
                    action = item["decision"]
                    if action in counts:
                        counts[action] += 1
                    else:
                        other += 1
            row = {"Esperimento": "Bottom-up originale" if entry["method"] == "bottom_up"
                   else entry["config_id"], "Dataset": entry["dataset"], "LLM": model,
                   "Ripetizione": entry["repetition"],
                   "Esempi evaluator": settings.get("evaluator_few_shot_examples"),
                   "Temperatura evaluator": settings.get("evaluator_temperature"),
                   "Iterazioni esaminate": len(traces), "Risposte globali valide": valid,
                   "Risposte globali non valide": invalid, "Iterazioni senza risposta registrata": missing,
                   "Tentativi scartati": rejected_attempts,
                   "Totale decisioni valide": sum(counts.values()) + other,
                   "Altre decisioni": other, "Policy": settings.get("pipeline_policy_version"),
                   "Stato": "Nessuna traccia globale" if not valid and not invalid else
                       "Traccia parziale" if missing else "Disponibile",
                   "Suite": entry["suite_id"], "Baseline": entry["baseline_key"]}
            row.update({labels[action]: counts[action] if valid or invalid else None for action in actions})
            if not valid and not invalid:
                row["Totale decisioni valide"] = None
                row["Altre decisioni"] = None
                row["Tentativi scartati"] = None
            rows.append(row)
    columns = ["Esperimento", "Dataset", "LLM", "Ripetizione", "Esempi evaluator",
        "Temperatura evaluator", "KEEP", "REWRITE", "REMOVE", "DUPLICATE_REMOVAL",
        "Totale decisioni valide", "Iterazioni esaminate", "Risposte globali valide",
        "Risposte globali non valide", "Tentativi scartati", "Iterazioni senza risposta registrata",
        "Altre decisioni", "Stato", "Policy", "Suite", "Baseline"]
    return pd.DataFrame(rows, columns=columns).sort_values(
        ["Esperimento", "Dataset", "Ripetizione", "LLM"], ignore_index=True)


def final_decision_counts_table(collection):
    """Count approved actions separately from the success of their application."""
    labels = {"KEEP_ORIGINAL_HIGH_LEVEL_GOAL": "KEEP",
              "REWRITE_ORIGINAL_HIGH_LEVEL_GOAL": "REWRITE",
              "REMOVE_ORIGINAL_HIGH_LEVEL_GOAL": "REMOVE", "DUPLICATE_REMOVAL": "DUPLICATE_REMOVAL"}
    configs = {f"{c['suite_id']}/{c['config_id']}": c for c in collection["configurations"]}
    rows = []
    for entry in collection["entries"]:
        if entry["method"] == "top_down":
            continue
        cycle = entry["payload"].get("globalGoalCycle", {})
        settings = {**configs.get(entry["method"], {}), **cycle.get("configuration", {})}
        traces = [t for t in cycle.get("iterations", []) if t.get("checkpoint_complete", True)]
        for scope, selected in (("Intera run", traces), ("Ultima iterazione", traces[-1:])):
            counts = dict.fromkeys(["KEEP", "REWRITE", "REMOVE", "DUPLICATE_REMOVAL", "INCONCLUSIVE",
                "Totale decisioni", "Riparazioni completate", "Riparazioni rifiutate", "Rami rimossi"], 0)
            for trace in selected:
                outcomes = trace.get("structural_action_outcomes", {})
                for bid, decision in trace.get("evaluations", {}).items():
                    action = decision.get("decision")
                    if action is not None and action not in labels:
                        raise ValueError(f"Unsupported decision in schema-3 result: {action}")
                    counts[labels[action] if action else "INCONCLUSIVE"] += 1
                    counts["Totale decisioni"] += 1
                    outcome = outcomes.get(bid)
                    for expected, column in (("HLG_LLG_HANDOFF_COMPLETED", "Riparazioni completate"),
                        ("REJECTED_STATE_PRESERVED", "Riparazioni rifiutate"),
                        ("REMOVED_WITH_CHILDREN", "Rami rimossi")):
                        counts[column] += outcome == expected
            rows.append({"Esperimento": "Bottom-up originale" if entry["method"] == "bottom_up"
                         else entry["config_id"], "Dataset": entry["dataset"],
                "Ripetizione": entry["repetition"], "Ambito": scope,
                "Esempi evaluator": settings.get("evaluator_few_shot_examples"),
                "Temperatura evaluator": settings.get("evaluator_temperature"),
                "Iterazioni considerate": len(selected), **{k: v if selected else None for k, v in counts.items()},
                "Stato": "Disponibile" if selected else "Nessuna traccia",
                "Suite": entry["suite_id"], "Baseline": entry["baseline_key"]})
    columns = ["Esperimento", "Dataset", "Ripetizione", "Ambito", "Esempi evaluator",
        "Temperatura evaluator", "Iterazioni considerate", "KEEP", "REWRITE", "REMOVE",
        "DUPLICATE_REMOVAL", "INCONCLUSIVE", "Totale decisioni", "Riparazioni completate",
        "Riparazioni rifiutate", "Rami rimossi", "Stato", "Suite", "Baseline"]
    return pd.DataFrame(rows, columns=columns).sort_values(
        ["Esperimento", "Ambito", "Dataset", "Ripetizione"], ignore_index=True)


def export_decisions_by_configuration(report_dir, tables):
    """Keep combined exports plus individual configuration files for decision tables."""
    root = Path(report_dir) / "decisions_by_configuration"
    for name, table in tables.items():
        for (suite, config), frame in table.groupby(["Suite", "Esperimento"], dropna=False, sort=False):
            identity = f"{suite}/{config}"
            slug = "".join(c if c.isalnum() or c in "-_." else "_" for c in str(config))[:80]
            directory = root / f"{slug}_{digest(identity.encode())[:8]}"
            directory.mkdir(parents=True, exist_ok=True)
            frame.to_csv(directory / f"{name}.csv", index=False, encoding="utf-8-sig")
            frame.to_html(directory / f"{name}.html", index=False, na_rep="-")


In [ ]:
collection = collect_outputs(PROJECT_ROOT, SUITE_DIRS)
# Considera solo le baseline della campagna selezionata, anche per gli originali.
suite_names = {p.name for p in SUITE_DIRS}
planned_keys = {(r["dataset"], r["baseline_key"]) for r in collection["availability"]
                if r.get("suite_id") in suite_names}
collection["entries"] = [e for e in collection["entries"]
                         if (e["dataset"], e["baseline_key"]) in planned_keys]
collection["availability"] = [r for r in collection["availability"]
    if (r.get("dataset"), r.get("baseline_key")) in planned_keys]
print("Run completate da valutare:", len(collection["entries"]))
print("Dataset previsti:", len(planned_keys))


## Metriche sugli output finali

BERT e preprocessing NLTK della valutazione originale; matching uno-a-uno senza soglia. La prima esecuzione richiede che modello e risorse siano disponibili.


In [ ]:
from src.evaluation.goal_evaluator import GoalEvaluator

evaluator = GoalEvaluator(model_name="bert-base-uncased", preprocess=True)
METRIC_CONFIGURATION = {
    "model": "bert-base-uncased",
    "resolved_revision": getattr(evaluator.model.config, "_commit_hash", None),
    "goal_text": "description", "preprocessing": "original NLTK",
    "matching": "maximum-weight one-to-one; every assigned cosine contributes",
    "similarity_threshold": None,
    "aggregation": "Arithmetic mean of final-run P, R and F1; repetitions first within dataset, then equal dataset weights",
    "delta_policy": "Experiment mean minus top-down mean on identical paired dataset/baseline cohort; absolute difference",
    "iteration_policy": "Only finalHighLevelGoals and finalLowLevelGoals; no intermediate checkpoint scoring",
}
metrics, matches = evaluate_final_outputs(collection, GROUNDTRUTH_BY_NAME, evaluator.score_goals)
comparison, common_cohort = final_comparison_table(
    metrics, collection["configurations"], collection["availability"])

dataset_comparison = dataset_comparison_table(
    metrics, collection["configurations"], collection["availability"])


## Tabella unica delle medie finali

Le sei colonne Delta riportano la differenza tra ciascuna media sperimentale e quella top-down sulla stessa coorte: Delta = esperimento - top-down. Sono differenze assolute nella scala delle metriche, non variazioni percentuali. La riga top-down ha delta zero quando valutabile; risultati mancanti mantengono anche il delta mancante.


In [ ]:
# Tabella riepilogativa: valori tra 0 e 1; celle vuote = risultato non disponibile.
metric_columns = [f"{level} {metric}" for level in ("HLG", "LLG") for metric in ("Precision", "Recall", "F1")]
delta_columns = [f"{level} Delta {metric}" for level in ("HLG", "LLG") for metric in ("Precision", "Recall", "F1")]
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(comparison.style.format({**{c: "{:.4f}" for c in metric_columns}, **{c: "{:+.4f}" for c in delta_columns}}, na_rep="-").hide(axis="index"))
print("Dataset comuni:", ", ".join(sorted(name for name, _ in common_cohort)) or "nessuno")
print("Le medie usano gli output finali, non la media delle iterazioni intermedie.")
if any(comparison.Stato != "Disponibile"):
    print("Campagna incompleta o coorte ridotta: consultare Stato e numerosita nella tabella.")
print("Delta = esperimento - top-down, sugli stessi dataset: positivo indica incremento, negativo diminuzione.")


## Confronto per esperimento e dataset

Una sola tabella contiene tutte le configurazioni e i dataset, con le metriche HLG e LLG e i delta appaiati: Delta = esperimento - top-down dello stesso dataset e della stessa baseline. Le metriche usano il matching BERT uno-a-uno gia calcolato, senza nuove inferenze. Le eventuali ripetizioni vengono mediate separatamente per Precision, Recall e F1.

Questa tabella mostra tutti i dataset pianificati, senza limitarli alla coorte comune della tabella riepilogativa. Risultati mancanti rimangono vuoti; Stato e Ripetizioni valutate indicano la copertura. I delta mancanti non sono sostituiti con zero.


In [ ]:
# Suite e hash della baseline sono conservati negli export per la provenienza.
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(dataset_comparison.drop(columns=["Suite", "Baseline"]).style.format(
        {**{c: "{:.4f}" for c in metric_columns},
         **{c: "{:+.4f}" for c in delta_columns}}, na_rep="-").hide(axis="index"))
print("Delta per dataset = esperimento - top-down; positivo indica incremento, negativo diminuzione.")


## Convergenza e numero di goal

Una riga per dataset, esperimento e ripetizione: convergenza, motivo di arresto, iterazioni completate e numero di HLG e LLG nella baseline top-down e nella gerarchia finale bottom-up. I delta dei conteggi sono bottom-up meno top-down.

Le ripetizioni restano separate. Le run completate senza convergenza mantengono i conteggi finali; per run non completate i risultati finali e la convergenza restano non disponibili, senza utilizzare checkpoint intermedi. Una collezione finale vuota vale zero; un risultato mancante rimane vuoto.


In [ ]:
convergence_goal_counts = convergence_goal_counts_table(collection)
count_columns = ["Ripetizione", "Iterazioni completate", "HLG top-down", "HLG bottom-up",
                 "LLG top-down", "LLG bottom-up"]
count_deltas = ["Delta numero HLG", "Delta numero LLG"]
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(convergence_goal_counts.drop(columns=["Suite", "Baseline"]).style.format(
        {**{c: "{:.0f}" for c in count_columns},
         **{c: "{:+.0f}" for c in count_deltas}}, na_rep="-").hide(axis="index"))


## Decisioni dei singoli LLM per progetto e configurazione

Una riga per esperimento, dataset, ripetizione e LLM, con il numero di esempi e la temperatura effettivamente registrati. Le colonne KEEP, REWRITE, REMOVE e DUPLICATE_REMOVAL contano le decisioni dei singoli evaluator nelle risposte globali valide, sommando tutte le iterazioni completate della run. Uno stesso goal puo essere valutato piu volte: i conteggi rappresentano decisioni per ramo e iterazione, non goal distinti.

Sono proposte dei singoli modelli, non interventi applicati dall'orchestratore. Le risposte scartate non contribuiscono alle quattro azioni; risposte globali non valide e tentativi scartati sono riportati separatamente. Un retry scartato viene contato una volta per modello e iterazione, senza moltiplicarlo per il numero di branch. Le run non completate non entrano in questa analisi, coerentemente con la raccolta degli output finali. Eventuali tracce mancanti sono segnalate.

La visualizzazione e suddivisa in una tabella per configurazione; i file completi mantengono tutte le configurazioni e vengono affiancati dagli export separati.


In [ ]:
evaluator_decision_counts = evaluator_decision_counts_table(collection)
llm_count_columns = ["Ripetizione", "Esempi evaluator", "KEEP", "REWRITE", "REMOVE", "DUPLICATE_REMOVAL",
    "Totale decisioni valide", "Iterazioni esaminate", "Risposte globali valide",
    "Risposte globali non valide", "Tentativi scartati", "Iterazioni senza risposta registrata", "Altre decisioni"]
for (suite, config), frame in evaluator_decision_counts.groupby(["Suite", "Esperimento"], dropna=False, sort=False):
    settings = frame.iloc[0]
    print(f"Configurazione: {config} | esempi: {settings['Esempi evaluator']} | temperatura: {settings['Temperatura evaluator']}")
    with pd.option_context("display.max_columns", None, "display.max_rows", None):
        display(frame.drop(columns=["Suite", "Baseline", "Esperimento"]).style.format(
            {**{c: "{:.0f}" for c in llm_count_columns}, "Temperatura evaluator": "{:.1f}"},
            na_rep="-").hide(axis="index"))


## Decisioni aggregate del ciclo per configurazione

Le tabelle distinguono le quattro azioni approvate e i rami inconcludenti, separatamente per intera run e ultima iterazione completata. L'assenza di maggioranza e rappresentata da `decision=null`, non da una quinta azione LLM. La duplicazione mantiene il valore esatto `DUPLICATE_REMOVAL`.

Gli esiti applicativi sono separati: una riscrittura approvata ma rifiutata durante la riparazione resta una proposta di riscrittura, con esito di fallimento. I conteggi si riferiscono a ramo e iterazione, non a goal unici o chiamate ai modelli. Sono supportate soltanto nuove run con schema 3; le colonne delle policy storiche sono eliminate.


In [ ]:
final_decision_counts = final_decision_counts_table(collection)
final_count_columns = ["Ripetizione", "Esempi evaluator", "Iterazioni considerate", "KEEP", "REWRITE", "REMOVE",
    "DUPLICATE_REMOVAL", "INCONCLUSIVE", "Totale decisioni", "Riparazioni completate", "Riparazioni rifiutate",
    "Rami rimossi"]
for (suite, config), frame in final_decision_counts.groupby(["Suite", "Esperimento"], dropna=False, sort=False):
    settings = frame.iloc[0]
    print(f"Configurazione: {config} | esempi: {settings['Esempi evaluator']} | temperatura: {settings['Temperatura evaluator']}")
    for scope, scoped in frame.groupby("Ambito", sort=False):
        print(scope)
        with pd.option_context("display.max_columns", None, "display.max_rows", None):
            display(scoped.drop(columns=["Suite", "Baseline", "Esperimento", "Ambito"]).style.format(
                {**{c: "{:.0f}" for c in final_count_columns}, "Temperatura evaluator": "{:.1f}"},
                na_rep="-").hide(axis="index"))


## Salvataggio

`final_comparison.csv` e `final_comparison.html` contengono il riepilogo esistente. `dataset_comparison.csv` e `dataset_comparison.html` contengono la tabella unica per esperimento e dataset, incluse metriche, delta e provenienza. I CSV di dettaglio conservano anche copertura e metriche per singola run.

`convergence_goal_counts.csv` e `convergence_goal_counts.html` contengono convergenza e conteggi dei goal per dataset, esperimento e ripetizione.

`evaluator_decision_counts.csv` e `evaluator_decision_counts.html` contengono i conteggi delle decisioni per LLM, progetto e configurazione, con diagnostica e provenienza.

`final_decision_counts.csv` e `.html` riportano le decisioni aggregate e gli esiti applicativi. La cartella `decisions_by_configuration/` conserva inoltre entrambe le tabelle decisionali in file separati per configurazione.


In [ ]:
# Verifica che gli input non siano cambiati durante il calcolo.
changed = [str(p) for p, checksum in collection["input_hashes"].items()
           if not p.exists() or digest(p.read_bytes()) != checksum]
if changed:
    raise RuntimeError(f"Input modificati durante la valutazione; rieseguire il notebook: {changed}")
REPORT_DIR.mkdir(parents=True, exist_ok=False)
comparison.to_csv(REPORT_DIR / "final_comparison.csv", index=False, encoding="utf-8-sig")
comparison.to_html(REPORT_DIR / "final_comparison.html", index=False, float_format=lambda x: f"{x:.4f}", na_rep="-")
dataset_comparison.to_csv(REPORT_DIR / "dataset_comparison.csv", index=False, encoding="utf-8-sig")
dataset_comparison.to_html(REPORT_DIR / "dataset_comparison.html", index=False, float_format=lambda x: f"{x:.4f}", na_rep="-")
convergence_goal_counts.to_csv(REPORT_DIR / "convergence_goal_counts.csv", index=False, encoding="utf-8-sig")
convergence_goal_counts.to_html(REPORT_DIR / "convergence_goal_counts.html", index=False, na_rep="-")
evaluator_decision_counts.to_csv(REPORT_DIR / "evaluator_decision_counts.csv", index=False, encoding="utf-8-sig")
evaluator_decision_counts.to_html(REPORT_DIR / "evaluator_decision_counts.html", index=False, na_rep="-")
final_decision_counts.to_csv(REPORT_DIR / "final_decision_counts.csv", index=False, encoding="utf-8-sig")
final_decision_counts.to_html(REPORT_DIR / "final_decision_counts.html", index=False, na_rep="-")
export_decisions_by_configuration(REPORT_DIR, {
    "evaluator_decision_counts": evaluator_decision_counts,
    "final_decision_counts": final_decision_counts})
metrics.to_csv(REPORT_DIR / "final_metrics.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(collection["availability"]).to_csv(REPORT_DIR / "availability.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(collection["configurations"]).to_csv(REPORT_DIR / "configurations.csv", index=False, encoding="utf-8-sig")
manifest = {
    "suite_id": SUITE_DIRS[0].name, "metric": METRIC_CONFIGURATION,
    "common_cohort": [{"dataset": d, "baseline_key": b} for d, b in sorted(common_cohort)],
    "sources": {str(p): h for p, h in collection["input_hashes"].items()},
    "ground_truth_sha256": digest(json.dumps(GROUNDTRUTH_BY_NAME, sort_keys=True, ensure_ascii=False).encode()),
    "status_policy": "Only completed runs, including nonconvergent; missing/failed runs never replaced by zero",
    "dataset_table_policy": "All planned dataset/baseline pairs; final repetitions averaged within each method; deltas against exact matching top-down baseline; missing values preserved",
    "convergence_table_policy": "Per run; final HLG/LLG counts paired with frozen top-down baseline; repetitions separate; unfinished runs have no final counts or convergence",
    "evaluator_decision_table_policy": "Completed runs; accepted global-response decisions summed across completed iterations per LLM/dataset/configuration/repetition; rejected attempts counted once per global request, not per branch; no applied-action inference",
    "final_decision_table_policy": "Recorded orchestrator decisions and applied branch outcomes; whole run and last iteration separately; per-configuration displays and exports",
    "pairing": "Identical datasets and frozen baselines across methods with available final outputs",
}
(REPORT_DIR / "evaluation_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Tabella salvata:", REPORT_DIR / "final_comparison.csv")
print("Tabella per dataset salvata:", REPORT_DIR / "dataset_comparison.csv")
print("Convergenza e conteggi salvati:", REPORT_DIR / "convergence_goal_counts.csv")
print("Decisioni per LLM salvate:", REPORT_DIR / "evaluator_decision_counts.csv")
print("Decisioni finali e tabelle per configurazione salvate:", REPORT_DIR / "decisions_by_configuration")
